# Exploratory Data Analysis — Breast Cancer Wisconsin (Diagnostic)

**Course:** UE 5BM753 — Supervised and Unsupervised Learning for Biology
**Part of:** the Breast Cancer Wisconsin (Diagnostic) final report, split into
per-step notebooks. See `00_Final_Report_.ipynb` for the combined narrative and
`README.md` for the overall conclusion.

**Question addressed here:** what do the features look like, and which ones look useful?

Builds on `02_QC.ipynb`: the data is already checked clean there. This notebook takes a first visual look, to prepare the modeling steps that follow.

## Setup

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

sys.path.append(os.path.join("..", "01_scripts"))
from dataset import load_data, get_feature_columns
from plots import (plot_class_balance, plot_correlation_heatmap, get_top_correlated_pairs,
                    plot_boxplots_by_diagnosis, plot_pca_projection, plot_pca_scree)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid", font_scale=1.0)
plt.rcParams["figure.dpi"] = 110

DATA_PATH = os.path.join("..", "02_data", "raw", "10_Breast_Cancer_Wisconsin_Diagnostic.csv")
RESULTS_DIR = os.path.join("..", "04_results")
FIGURES_DIR = os.path.join("..", "05_figures")
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

df = load_data(DATA_PATH)
features = get_feature_columns(df)

## Exploratory Data Analysis (EDA)

Now that the quality checks are done, we take a first visual look at the data. This
step helps us understand the data before choosing any method. It also raises questions
that the rest of the analysis will answer.

Four views are shown below. Each one prepares a later step.

- **Class balance**. This shows how many samples are benign vs malignant. It motivates
  the choice of evaluation scores used later, when models are compared.
- **Correlation heatmap**. This shows which of the 30 raw features carry the same
  information. It motivates the feature selection step later in the analysis.
- **Boxplots by diagnosis**. This shows which features visually separate benign from
  malignant samples. These features carry useful signal for the modeling steps that
  follow.
- **PCA projection and variance explained**. This gives a quick visual check of how
  separable the two classes are. The 2D coordinates are reused later to visualize the
  clustering results.

**Class balance.** The dataset is moderately imbalanced: about 63% benign and 37%
malignant. Because of this imbalance, accuracy alone is not a reliable way to judge a
model. A model could reach high accuracy just by guessing "benign" most of the time,
while still missing many malignant cases. This motivates using scores that
specifically account for how well malignant cases are detected, later when models are
compared.

In [ ]:
plot_class_balance(df, os.path.join(FIGURES_DIR, "final_01_class_balance.png"))

**Correlation heatmap.** Feature pairs with |r| > 0.95 are highly redundant. They carry
almost the same information. This motivates removing some of them in the feature
selection step later. Several of the 30 raw features can likely be dropped without
losing signal.

In [ ]:
plot_correlation_heatmap(df, features, os.path.join(FIGURES_DIR, "final_02_correlation_heatmap.png"))

top_pairs = get_top_correlated_pairs(df, features, threshold=0.95)

pd.DataFrame(top_pairs, columns=["feature_1", "feature_2", "pearson_r"]).to_csv(
    os.path.join(RESULTS_DIR, "01_correlated_feature_pairs.csv"), index=False)

print(f"Feature pairs with |correlation| > 0.95 ({len(top_pairs)} pairs found):\n")
for f1, f2, corr_val in top_pairs[:10]:
    print(f"  {f1:28s} <-> {f2:28s}  r = {corr_val:.3f}")


**Boxplots by diagnosis (mean features).** Only the 10 `_mean` features are shown here,
one per base measurement. This keeps the view readable: showing all 30 features
(`_mean`, `_se`, `_worst`) at once would be redundant, since many `_worst` values are
highly correlated with their `_mean` counterpart, as seen in the correlation heatmap
above. Features whose distributions clearly separate benign from malignant carry
useful signal. The more separated the two boxes are, the more useful that feature is
likely to be for the feature selection and modeling steps that follow.

In [ ]:
mean_features = [f for f in features if f.endswith("_mean")]
plot_boxplots_by_diagnosis(df, mean_features, os.path.join(FIGURES_DIR, "final_03_boxplots_mean_features.png"))


**PCA projection and variance explained.** Principal Component Analysis (PCA)
compresses many features into a few new ones, called components, while keeping as much
of the original information as possible. Here, the 30 features are compressed into
just 2 components, which can be plotted on a simple 2D graph. This gives a quick visual
check of how well the two classes separate, before any real model is trained. The
scree plot shows how much of the total variance these first components actually
capture. The 2D coordinates are reused later to visualize the clustering results.

In [ ]:
X_scaled_full = StandardScaler().fit_transform(df[features].values)

pca_full, X_pca_full = plot_pca_projection(X_scaled_full, df["label"].values,
os.path.join(FIGURES_DIR, "final_04_pca_projection.png"), random_state=RANDOM_STATE,)

print(f"PC1 + PC2 explain {sum(pca_full.explained_variance_ratio_)*100:.1f}% of total variance\n")

cum_var = plot_pca_scree(X_scaled_full, os.path.join(FIGURES_DIR, "final_05_pca_scree.png"))
n_90 = int((cum_var < 0.90).sum() + 1)

print(f"{n_90} principal components are needed to reach 90% cumulative variance")


## Discussion

The correlation heatmap shows many of the 30 raw features are redundant, and the
boxplots show several features already separate benign from malignant well. Both
motivate the next step: `04_Initial_Screening.ipynb` compares standard models on all
30 features, before `05_Feature_Selection.ipynb` reduces them to a smaller,
non-redundant subset.